# SPOD to Sharepoint integration

- Prerequisites: 
  - [Office365-REST-Python-Client](https://pypi.org/project/Office365-REST-Python-Client/) installed. `pip install Office365-REST-Python-Client`
  - Anaconda packages: `pandas, openpyxl`
- Access to a [list(s)](https://support.microsoft.com/en-us/office/introduction-to-lists-0a1c3ace-def0-44af-b225-cfa8d92c52d7) in Sharepoint / Office365.

## Structure

1. Define mapping between SPOD (json) and columns in the list
1. Ensure list columns conform mapping. Automatically update if need.
1. Scan current content and preserve it in a table (Pandas)
1. Show delta / changes that will occur on upload
1. Upload changes


## Configuration

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

CONFIGURATION = 'sharepoint.yaml'

apply_changes = True

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
configfile = Path(CONFIGURATION)
assert configfile.is_file(), f"Cannot find configuration file '{configfile.resolve()}'"

with open(configfile, 'r') as src:
    configuration = yaml.safe_load(src)
assert configuration['sharepoint'] is not None
spconf = configuration['sharepoint']
print(f"Loaded configuration for sharepoint acces with user '{spconf['credentials']['username']}' from {configfile}")

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.sharepoint.list_publisher import update_structure, update_content, load_content

### Check SPOD

In [ ]:
spod_file = Path(configuration['spod'])
assert spod_file.is_file(), f"SPOD source missing: {spod_file.resolve()}"

In [ ]:
with open(spod_file, 'r') as src:
    spod = json.load(src)
print(f"Version {spod['_imprint_']}")
mapdict = {}
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    print(f"- {key}: {len(spod[key])}")
    mapdict[key] = entry[key]['title']

## Access to Sharepoint site using Office365-REST-Python-Client library

In [ ]:
try:
    from office365.sharepoint.lists.list import List
    from office365.runtime.auth.user_credential import UserCredential
    from office365.sharepoint.client_context import ClientContext
except:
    print("Office356 API library is missing. Install it with 'pip install Office365-REST-Python-Client'")
    raise

In [ ]:
credentials = UserCredential(spconf['credentials']['username'], spconf['credentials']['password'])
ctx = ClientContext(spconf['site']).with_credentials(credentials)

In [ ]:
lists_available = ctx.lists.get().execute_query()
assert len(lists_available) > 0, f"Expecting more than 0 lists"
print(f"Found {len(lists_available)} lists in site {spconf['site']}")
print(f"Mapping exists for {len(mapdict)} tables: {list(mapdict.values())}")
for spl in lists_available:
    mapped = '✅' if spl.title in mapdict.values() else ''
    print(f"- {spl.title} {mapped}")

## List structure definition

In [ ]:
mappings = {}

In [ ]:
### Define Translation
deflang = 'en'

def tr(item):
    if isinstance(item, dict) and len(item) > 0:
        en = item.get(deflang)
        if en is not None:
            return en
        else:
            return next(iter(item.values()))
    if isinstance(item, str):
        return item
    
    if isinstance(item, list):
        return tr(item[0])
    return ''

### Defintion of the 'Entity' list

In [ ]:
entity_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
        'CanBeDeleted': False
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {'CanBeDeleted': False}}),
    ('Synonyms', {'value': lambda e: tr(e['synonyms']), 'properties': {}}),
    ('Diagrams', {'value': lambda e: str(e['diagrams+']), 'properties': {}}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'EnforceUniqueValues': True,
        'CanBeDeleted': False,
        'Filterable': True,
        'Sortable': True,
        'Indexed': True,
        'FieldTypeKind': 2,
        'MaxLength': 10,
        'Required': True,
    }}),
    #   ('Documentation Link', {'value': lambda e: './bla.html', 'properties': {'FieldType': 11}}),
]
mappings['entities'] = entity_mapping

### Definition of the 'Attribute' list
This list contains **all** attributes of the IM

In [ ]:
attribute_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
        'CanBeDeleted': False
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {'CanBeDeleted': False}}),
    ('Synonyms', {'value': lambda e: tr(e['synonyms']), 'properties': {}}),
    ('Type', {'value': lambda a: a['basetype+'], 'properties': {
        'Description': 'Datatype of the attribute',
        'CanBeDeleted': False,
        'Filterable': True,
        'Sortable': True,
        'Indexed': True,
        'FieldType': 2,
        'MaxLength': 50,
        'Required': False,
    }}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'CanBeDeleted': False,
        'EnforceUniqueValues': True,
        'Filterable': True,
        'Indexed': True,
        'Sortable': True,
        'FieldType': 2,
        'MaxLength': 10,
        'Required': True,
    }}),
    #   ('Documentation Link', {'value': lambda e: './bla.html', 'properties': {'FieldType': 11}}),
]
mappings['attributes'] = attribute_mapping

# Preparation steps

## Preparing list structure 

In [ ]:
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        sp_list = ctx.lists.get_by_title(entry[key]['title'])
        result = update_structure(sp_list, mapping, write=apply_changes)
        print(f"Result:\n{os.linesep.join(result)}")
    else:
        logging.warning(f"No mapping for {key}")

# Synchronize content
1. Read content, store it to local backup
2. Apply changes if any

In [ ]:
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        sp_list = ctx.lists.get_by_title(entry[key]['title'])
        content = load_content(sp_list)
        print(f"List {entry[key]['title']} currently contains {len(content)} entries")
        new, updated, deleted = update_content(sp_list, mapping, spod[key], content)
        #print(f"Result:\n{os.linesep.join(result)}")
        print(f"Creating: {len(new)}, updating: {len(updated)}, deleting: {len(deleted)}")
    else:
        logging.warning(f"No mapping for {key}")